# День 1 · streaming и TTFT

Без стрима клиент ждёт весь ответ целиком. Со стримом (`stream=True`) провайдер отдаёт ответ
кусками по мере генерации — так работает любой чат в браузере: текст печатается сразу, а не
появляется одним куском через несколько секунд. Ниже измеряем **TTFT** (time to first token) —
время от отправки запроса до первого символа ответа: для чата это главная метрика ощущаемой
скорости, для фоновых задач важнее суммарная пропускная способность (tok/s), не TTFT.

In [ ]:
import time

import labkit  # noqa: F401  читает .env
from client import make_client, MODEL

# --- НАСТРОЙКИ ---
PROMPT = "Объясни в пяти предложениях, что такое RAG, для DevOps-инженера."
MAX_TOKENS = 300

client = make_client()
t0 = time.perf_counter()      # момент отправки запроса
first = None                  # момент первого токена (для TTFT)
usage = None                  # счётчики токенов, приходят в последнем чанке

Внутри провайдера каждый новый токен считается по уже обработанным токенам контекста — это и есть
**KV-cache**: сохранённые представления attention для всех предыдущих токенов, чтобы не пересчитывать
их заново на каждом шаге генерации. Это невидимо снаружи (у нас нет доступа к KV-cache напрямую),
но именно оно определяет, почему TTFT растёт с длиной подсказки, а генерация после первого токена
идёт быстрее прироста контекста — векторов уже посчитано.

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    stream=True,                                  # ответ приходит кусками, а не целиком
    stream_options={"include_usage": True},       # попросить usage в конце стрима
    max_tokens=MAX_TOKENS,
    messages=[{"role": "user", "content": PROMPT}],
)
for event in stream:                              # каждый event — один кусок ответа
    if event.usage:
        usage = event.usage                       # последний чанк: только usage, без текста
    if not event.choices:
        continue
    delta = event.choices[0].delta.content or ""  # новые символы в этом куске
    if delta and first is None:
        first = time.perf_counter()               # первый токен пришёл
    print(delta, end="", flush=True)              # печатаем сразу, как чат в браузере

`first - t0` — и есть TTFT: сеть, очередь провайдера и обработка промпта (prefill) до появления
первого символа. `t1 - first` — время самой генерации; поделив на число токенов, получаем tok/s.

In [ ]:
t1 = time.perf_counter()
if first is None:
    raise RuntimeError("стрим не содержал текста: проверь отказ и finish_reason")
print(f"\n\nTTFT: {first - t0:.2f}s | всего: {t1 - t0:.2f}s", end="")     # TTFT — время до первого токена
if usage and first:
    print(f" | out={usage.completion_tokens} → {max(0, usage.completion_tokens - 1) / max(t1 - first, 1e-9):.1f} tok/s")  # скорость генерации
else:
    print(" | usage в стриме не пришёл — провайдер не поддерживает include_usage")